In [1]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv()
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))
#print isi API nya
print(ALPHA_KEY)

Loaded ALPHAVANTAGE_API_KEY? True
QNF78IHU7X1LZ9KJ


In [2]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

In [5]:
SYMBOL = "IQQBDA"
use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={SYMBOL}&apikey={ALPHA_KEY}"
    r = requests.get(url)
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    assert key, f"Unexpected response keys: {list(js.keys())}"
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])
else:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d").reset_index()[['Date','Adj Close']]
    df_api.columns = ['date','close']

# 添加 ticker 列
df_api['ticker'] = SYMBOL

# 重新排列列的顺序，将 ticker 放在中间
cols = ['date', 'ticker', 'close']
df_api = df_api[cols]

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','ticker','close'], dtypes_map={'date':'datetime64[ns]','close':'float'})
print(msgs)

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Saved: data\raw\api_source-alpha_symbol-IQQBDA_20250821-174446.csv


In [9]:
# 修改后的代码 - 获取并转换数据格式
SYMBOLS = ["AAPL", "VCNIX"]  # 确保使用正确的纳斯达克100符号
use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

# 创建一个空列表来存储所有数据
all_data = []

for symbol in SYMBOLS:
    print(f"Fetching data for {symbol}...")
    
    if use_alpha:
        # 添加延迟以避免API限制
        if len(all_data) > 0:
            time.sleep(5)  # 等待15秒
        
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_KEY}"
        r = requests.get(url)
        js = r.json()
        
        # 检查API响应
        if "Error Message" in js:
            print(f"Error fetching {symbol}: {js['Error Message']}")
            continue
        
        if "Note" in js:
            print(f"Note for {symbol}: {js['Note']}")
            continue
            
        key = [k for k in js.keys() if "Time Series" in k]
        if not key:
            print(f"Unexpected response for {symbol}: {list(js.keys())}")
            continue
            
        series = js[key[0]]
        df = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
        
        # 提取收盘价列
        df = df[['date', '4. close']].rename(columns={'4. close': 'close'})
        df['date'] = pd.to_datetime(df['date'])
        df['close'] = pd.to_numeric(df['close'])
        
    else:
        # 使用yfinance回退方案
        import yfinance as yf
        # 对于指数，yfinance使用^前缀
        yf_symbol = symbol if symbol != "IXIC" else "^IXIC"
        df = yf.download(yf_symbol, period="6mo", interval="1d").reset_index()[['Date', 'Adj Close']]
        df.columns = ['date', 'close']
    
    # 添加ticker列
    df['ticker'] = symbol
    
    # 添加到总数据列表
    all_data.append(df)
    
    print(f"Successfully fetched {len(df)} records for {symbol}")

# 合并所有数据
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # 将数据从长格式转换为宽格式
    wide_df = combined_df.pivot(index='date', columns='ticker', values='close').reset_index()
    
    # 重命名列
    wide_df.columns.name = None  # 移除列名
    wide_df = wide_df.rename(columns={'AAPL': 'aapl', 'IXIC': 'nasdaq100'})
    
    # 删除任何包含NaN的行（确保时间序列对齐）
    wide_df = wide_df.dropna()
    
    # 按日期排序
    wide_df = wide_df.sort_values('date').reset_index(drop=True)
    
    # 验证数据
    msgs = validate_df(
        wide_df, 
        required_cols=['date', 'aapl', 'nasdaq100'], 
        dtypes_map={'date': 'datetime64[ns]', 'aapl': 'float', 'nasdaq100': 'float'}
    )
    print("Wide format data validation:", msgs)
    
    # 保存转换后的数据
    fname = safe_filename(
        prefix="wide_format", 
        meta={"source": "alpha" if use_alpha else "yfinance", "symbols": "-".join(SYMBOLS)}
    )
    out_path = DATA_RAW / fname
    wide_df.to_csv(out_path, index=False)
    print("Saved wide format data:", out_path)
    
    # 显示数据摘要
    print("\nData summary:")
    print(f"Total records: {len(wide_df)}")
    print(f"Date range: {wide_df['date'].min()} to {wide_df['date'].max()}")
    print("Sample data:")
    print(wide_df.head())
    
else:
    print("No data was fetched successfully.")

Using Alpha Vantage: True
Fetching data for AAPL...
Successfully fetched 100 records for AAPL
Fetching data for VCNIX...
Successfully fetched 100 records for VCNIX
Wide format data validation: {'missing_cols': "Missing columns: ['nasdaq100']", 'na_total': 'Total NA values: 0'}
Saved wide format data: data\raw\wide_format_source-alpha_symbols-AAPL-VCNIX_20250821-175922.csv

Data summary:
Total records: 100
Date range: 2025-03-28 00:00:00 to 2025-08-20 00:00:00
Sample data:
        date    aapl  VCNIX
0 2025-03-28  217.90  20.83
1 2025-03-31  222.13  20.83
2 2025-04-01  223.19  21.00
3 2025-04-02  223.89  21.16
4 2025-04-03  203.19  20.01
